# Set up

## Check configuration
Should return path to correct python version (from virtual environment)

In [ ]:
import sys
print(sys.executable)
# print('\n'.join(sys.path[:6]))

## Load libraries

In [ ]:
# Automatically reload modules before execution of each cell
# so when you edit src/mypackage/*.py in your editor and rerun cells, 
# changes appear immediately.
%reload_ext autoreload
%autoreload 2

# python
from __future__ import annotations
# from IPython.core.debugger import Pdb

# Standard library
from pathlib import Path

# Third-party
from pysdmx.model import FixedValueMap, ImplicitComponentMap, ValueMap, MultiValueMap, ComponentMap
from openpyxl import Workbook, load_workbook
import pandas as pd
import pysdmx as px
import pickle as pkl
from datetime import datetime


# Custom
## Functions
from tidysdmx import (
    filter_tidy_raw, 
    validate_dataset_local, 
    map_structures, 
    # infer_schema, 
    # infer_role_dimension, 
    apply_fixed_value_maps, 
    apply_implicit_component_maps, 
    build_date_pattern_map,
    build_value_map_list,
    # build_multi_value_map_list,
    build_representation_map,
    build_single_component_map,
    extract_component_ids,
    write_excel_mapping_template,
    build_structure_map,
    create_schema_from_table,
    build_structure_map_from_template_wb,
    build_multi_representation_map,
    apply_multi_component_map,
    standardize_output
)

## Define globals

In [ ]:
# CAUTION! FOR TESTING ONLY. DO NOT USE IN PRODUCTION.
# os.environ["PYTHONHTTPSVERIFY"] = "0"

# FMR and artefacts information
fmr_url = "https://fmrqa.worldbank.org/FMR/sdmx/v2"
# raw schema
raw_structure_agency = "WB"
raw_structure_id = "IFPRI_ASTI"
raw_structure_version = "1.0"
# dissemination schema
dis_structure_agency = "WB.DATA360"
dis_structure_id = "DS_DATA360"
dis_structure_version = "1.3"
# structure map
raw_structure_map = "SM_IFPRI_ASTI_TO_DATA360"

# Path to raw tidy data
# Path to raw data
path_to_raw_data = Path(
    "../../TMP/data/WB_ASPIRE"
)
path_to_xlsx_mapping = Path(
    "./data/MAPPING_TEMPLATE_EXAMPLE.xlsx"
)

## Initiate API client

In [ ]:
print(fmr_url)
client = px.api.fmr.RegistryClient(fmr_url)
client

# STEP 1 - Load raw data (in tidy format)


We start this notebook with tidy data that has already been fetch from the source and gone through basic cleaning and reshaping. This part (fetching, cleaining, reshaping) will always be speficic to a given dataset. It is therefore not possible to standardize this part of a pipeline, and it will not be covered in this notebook.

In [ ]:
df = pd.read_csv(path_to_data)
df.head()

# STEP 2 - Get metadata artefacts from FMR

## Get raw dataset schema
Information about the expected tidy data schema is stored in the FMR. We will fetch this information using the pysdmx library. This information will be used for early validation of the tidy data. That information could also be used to detect non-structural changes in the source data over time: Addition/removal of indicators for instance. This is not yet covered in this notebook. 

In [ ]:
raw_schema = client.get_schema("datastructure", agency=raw_structure_agency, id=raw_structure_id, version=raw_structure_version)
raw_schema

## Get structure map
Mapping information from raw data schema to dissemination schema is also being stored in FMR. That information can be easily retrieved using from the FRM API using `pysdmx`.

In [ ]:
sm = client.get_mapping("WB", "SM_IFPRI_ASTI_TO_DATA360")
sm

In [ ]:
tst_df=pd.DataFrame({
            "FREQ": ["A", "Q"],
            "AREA": ["USA", "FRA"],
            "INDICATOR": ["GDP", "POP"],
            "SEX": ["M", "F"],
            "AGE": ["Y18T65", "Y25T44"],
            "URBANISATION": ["URB", "RUR"],
            "UNIT_MEASURE": ["USD", "USD"],
            "COMP_BREAKDOWN_1": ["_Z", "_Z"],
            "OBS_VALUE": [100, 200],
            "TIME_PERIOD": ["2020", "2021"],
            "NOTE": ["A", "B"]
        })

## Get dissemination dataset schema

In [ ]:
dis_schema = client.get_schema("datastructure", agency=dis_structure_agency, id=dis_structure_id, version=dis_structure_version)
dis_schema

# STEP 3 - Check, filter, validate raw data

Information contained in the SDMX artefacts (schema + content constraints) can be used to validate raw data, flag new / removed indicators, and filter rows to be further processed. 

## Filter out rows that are not needed

In [ ]:
df = filter_tidy_raw(df=df, schema=raw_schema)
df.head()

## Validate cleaned-up raw data before mapping

In [ ]:
# errors = validate_dataset_local(df, schema=schema, sdmx_cols=[])
errors = validate_dataset_local(df, schema=raw_schema, sdmx_cols=[])

# STEP 4: Map data from raw to dissemination schema

In [ ]:
out = map_structures(df = df, structure_map = sm)
out.head()

# STEP 5: Validate output

In [ ]:
dis_errors = validate_dataset_local(df = out, schema = dis_schema)
dis_errors

In [ ]:
# TEMPORARY CHUNK
# out.to_csv('./data/formatted-data.csv', index=False)

# TESTING

In [ ]:
os.startfile(path_to_lorena_xls)

In [ ]:
# Get all Excel sheets
# wb=load_workbook(path_to_lorena_xls)
# Option 1: lightweight listing without parsing
# xf = pd.ExcelFile(path_to_lorena_xls, engine="openpyxl")
# sheet_names = xf.sheet_names
# sheet_names

# Option 2: parse all sheets into a dict; keys are sheet names
dfs = pd.read_excel(path_to_lorena_xls, sheet_name=None, engine="openpyxl")
list(dfs.keys())

In [ ]:
# functions to be implemented:
# _parse_info_sheet()
# _parse_comp_mapping()
# _parse_rep_mapping()

# tmp = pd.read_excel(path_to_lorena_xls, sheet_name='INFO', header=None, skiprows=5, usecols=range(3,5))
tmp=_parse_info_sheet(dfs)
sdmx_id=_extract_artefact_id(tmp, structure_type = "provision-agreement")

In [ ]:
# Load once; parse sheets later
xf = pd.ExcelFile(file_path, engine="openpyxl")

# Parse a specific sheet by name or index
df_sheet1 = xf.parse("Sheet1")      # or xf.parse(0)
df_other   = xf.parse("OtherSheet")

wb.sheetnames

In [ ]:
dfs = pd.read_excel(path_to_lorena_xls, sheet_name=None, engine="openpyxl")
tst=build_structure_map_from_template_wb(dfs)



In [ ]:
tst2=tst.component_maps[0]

In [ ]:
tst2.values

In [ ]:
data = {
            "source": ["BE", "FR"],
            "target": ["BEL", "FRA"],
            "valid_from": ["2020-01-01", None],
            "valid_to": ["2025-12-31", None]
        }
sample_df=pd.DataFrame(data)

In [ ]:
result = build_multi_representation_map(
            sample_df,
            id="MRM1",
            name="Country Multi Map",
            agency="ECB",
            source_cls=["urn:source:codelist"],
            target_cls=["urn:target:codelist"]
        )

In [ ]:
result

In [ ]:
tst=sm.multi_component_maps
tst=tst[0].values

In [ ]:
tst.maps

In [ ]:
tmp=os.environ.get('PYTHONBREAKPOINT')

In [ ]:
wb = Workbook()
# comp_mapping sheet
ws_comp = wb.create_sheet("comp_mapping")
ws_comp.append(["source", "target", "mapping_rules"])
ws_comp.append(["SRC1", "TGT1", "fixed:VAL1"])
ws_comp.append(["SRC2", "TGT2", "implicit"])
ws_comp.append(["SRC3", "TGT3", "TGT3"])  # representation map
# representation sheet for TGT3
ws_rep = wb.create_sheet("TGT3")
ws_rep.append(["source", "target", "valid_from", "valid_to"])
ws_rep.append(["A", "B", "", ""])

In [ ]:
workbook=wb

definitions = _extract_mapping_definitions(workbook)
    
maps_list = []
    
# 2. Convert Definitions to pysdmx Objects
for definition in definitions:
    if definition.map_type == "fixed":
        if not definition.source or not definition.source.strip():
            raise ValueError(f"Fixed value missing for {definition.target}")
        maps_list.append(
                build_fixed_map(target=definition.target, value=definition.fixed_value)
            )
            
    # elif definition.map_type == "implicit":
    #     if not definition.source or not definition.source.strip():
    #         raise ValueError(f"Source missing for implicit map {definition.target}")
    #     maps_list.append(
    #         build_implicit_component_map(source=definition.source, target=definition.target)
    #     )

In [ ]:
df = pd.DataFrame({
            "AREA": ["AAA", "BBB"],
            "NOTE": ["ccc", "ddd"]
        })

In [ ]:
multi_component_map=sm.maps[4]

In [ ]:
result = apply_multi_component_map(df, multi_component_map)

In [ ]:
with open(path_to_sm_fixture, 'rb') as f:
    sm_fxtr = pkl.load(f)



In [ ]:
sm_fxtr.maps

In [ ]:
sample_df=pd.DataFrame({
            "FREQ": ["A", "Q"],
            "AREA": ["USA", "FRA"],
            "INDICATOR": ["GDP", "POP"],
            "SEX": ["M", "F"],
            "AGE": ["Y18T65", "Y25T44"],
            "URBANISATION": ["URB", "RUR"],
            "UNIT_MEASURE": ["USD", "USD"],
            "COMP_BREAKDOWN_1": ["_Z", "_Z"],
            "OBS_VALUE": [100, 200],
            "TIME_PERIOD": ["2020", "2021"]
        })

In [ ]:
artefact_id = "WB:IFPRI_ASTI(1.0)"
result = standardize_output(sample_df, artefact_id=artefact_id, schema=raw_schema)

In [ ]:
comp_mapping = pd.DataFrame({"MAPPING_RULES": ["PREFIX "]})

errors = _collect_mapping_rules_errors(
    comp_mapping,
    valid_rules=["OK"],
    valid_prefixes=["PREFIX "]
)
errors

In [ ]:
def valid_mappings():
    """Fixture: Valid mappings dictionary with INFO, COMP_MAPPING, and REP_MAPPING sheets."""
    info_df = pd.DataFrame({"Key": ["dataflow"], "Value": ["AGENCY:DF_ID(1.0)"]})
    comp_df = pd.DataFrame({
        "SOURCE": ["SRC1", "SRC2", "SRC3"],
        # 123: ["SRC1", "SRC2", "SRC3"],
        "TARGET": ["TGT1", "TGT2", "TGT3"],
        "MAPPING_RULES": ["fixed:VAL1", "implicit", "other"]
    })
    rep_df = pd.DataFrame({
        "S:SRC3": ["A", "B"],
        "T:TGT3": ["X", "Y"]
    })
    return {"INFO": info_df, "COMP_MAPPING": comp_df, "REP_MAPPING": rep_df}

mappings = valid_mappings()
mappings


In [ ]:
structure_map = build_structure_map_from_template_wb(mappings)
structure_map